# 7. 现代卷积神经网络

在本章中的每一个模型都曾一度占据主导地位，其中许多模型都是ImageNet竞赛的优胜者。

> ImageNet竞赛自2010年以来，一直是计算机视觉中监督学习进展的指向标。

这些模型包括：

*   **AlexNet**。它是第一个在大规模视觉竞赛中击败传统计算机视觉模型的大型神经网络；

*   **使用重复块的网络（VGG）**。它利用许多重复的神经网络块；

*   **网络中的网络（NiN）**。它重复使用由卷积层和 $1 \times 1$ 卷积层（用来代替全连接层）来构建深层网络;

*   **含并行连结的网络（GoogLeNet）**。它使用并行连结的网络，通过不同窗口大小的卷积层和最大汇聚层来并行抽取信息；

*   **残差网络（ResNet）**。它通过残差块构建跨层的数据通道，是计算机视觉中最流行的体系架构；

*   **稠密连接网络（DenseNet）**。它的计算成本很高，但给我们带来了更好的效果。

虽然深度神经网络的概念非常简单——将神经网络堆叠在一起。但由于不同的网络架构和超参数选择，这些神经网络的性能会发生很大变化。本章介绍的神经网络是将人类直觉和相关数学见解结合后，经过大量研究试错后的结晶。 

---


## 7.1 深度卷积神经网络 (AlexNet)

> 如果说 LeNet 是神经网络的“童年”，那么 **AlexNet** 就是神经网络的“成年礼”。2012 年，AlexNet 在 ImageNet 竞赛中以绝对优势夺冠，彻底终结了传统计算机视觉的统治，开启了深度学习的黄金时代。

### 1. 核心动机：为什么 LeNet 不够了？

LeNet 在处理手写数字（28x28）时非常完美，但在面对真实世界的自然图像（如 ImageNet 的高清图）时遇到了瓶颈：

1.  **特征复杂度**：真实图像的纹理、光照和背景极其复杂，LeNet 的参数量不足以捕捉这些特征。
2.  **计算资源**：2012 年左右，GPU 算力开始爆发，使得训练更深、更宽的网络成为可能。
3.  **激活函数限制**：Sigmoid 在深层网络中极易导致**梯度消失**。

---

### 2. AlexNet 的五大核心创新

1.  **更深的网络结构**：8 层加权层（5 层卷积 + 3 层全连接）。
2.  **使用 ReLU 激活函数**：用 $\text{max}(0, x)$ 代替 Sigmoid。ReLU 在正区间导数为 1，彻底解决了深层网络的梯度消失问题，且计算速度极快。
3.  **使用 Dropout (暂退法)**：在最后的全连接层引入 Dropout（我们在 4.6 节学过），有效控制了巨大参数量带来的过拟合。
4.  **数据增广 (Data Augmentation)**：通过随机裁剪、翻转和亮度调整，人为增加训练样本，提高模型鲁棒性。
5.  **最大汇聚 (Max Pooling)**：用最大汇聚代替 LeNet 的平均汇聚，更能突出图像的显著特征。

---

### 3. 构建 AlexNet

AlexNet 的输入通常是 $224 \times 224$。由于 Fashion-MNIST 只有 $28 \times 28$，我们在加载数据时需要将其**强制放大**。


In [6]:
import torch
from torch import nn

def get_alexnet_model(num_classes: int = 10) -> nn.Sequential:
    """构建 AlexNet 模型。通过更深的卷积层和更轻的输出层来达到同样甚至更好的效果。
    
    结构参考 2012 年经典论文，针对 Fashion-MNIST 输出 10 类。
    """
    net = nn.Sequential(
        # 第一块：大尺寸卷积核提取粗略特征
        # 输入：(Batch, 1, 224, 224)
        nn.Conv2d(1, 64, kernel_size=11, stride=4, padding=2),  # -> (Batch, 64, 55, 55)
        nn.ReLU(),
        # 重叠汇聚：
        #   - 缓解过拟合;
        #   - 增大感受野，并避免信息通过更深的卷积层和更轻的输出层来达到同样甚至更好的效果。丢失过快
        nn.MaxPool2d(kernel_size=3, stride=2),                  # -> (Batch, 64, 27, 27)

        # 第二块：减小核尺寸，增加通道数
        # 输入：(Batch, 64, 27, 27)
        nn.Conv2d(64, 192, kernel_size=5, padding=2),           # -> (Batch, 192, 27, 27)
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2),                  # -> (Batch, 192, 13, 13)

        # 第三块：连续三个卷积层提取精细特征
        # 输入：(Batch, 192, 13, 13)
        nn.Conv2d(192, 384, kernel_size=3, padding=1),          # -> (Batch, 384, 13, 13)
        nn.ReLU(),
        nn.Conv2d(384, 256, kernel_size=3, padding=1),          # -> (Batch, 256, 13, 13)
        nn.ReLU(),
        nn.Conv2d(256, 256, kernel_size=3, padding=1),          # -> (Batch, 256, 13, 13)
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2),                  # -> (Batch, 256, 6, 6)

        # 第四块：全连接层 + Dropout
        nn.Flatten(),                                           # -> (Batch, 256 * 6 * 6)
        # 参数量主要集中在该全连接层
        nn.Linear(256 * 6 * 6, 4096),                           # -> (Batch, 4096)
        nn.ReLU(),
        # Dropout 抑制过拟合
        nn.Dropout(p=0.5),
        # 4096 -> 4096 是当时认为的力大砖飞，
        # 现在更倾向于通过更深的卷积层和更轻的输出层来达到同样甚至更好的效果。
        nn.Linear(4096, 4096),                                  # -> (Batch, 4096)
        nn.ReLU(),
        nn.Dropout(p=0.5),

        # 输出层
        nn.Linear(4096, num_classes)                            # -> (Batch, num_classes)
    )
    return net

# --- 验证维度流转 ---
X = torch.randn(1, 1, 224, 224)
net = get_alexnet_model()
for layer in net:
    X = layer(X)
    print(f"{layer.__class__.__name__:15} 输出形状: {X.shape}")

Conv2d          输出形状: torch.Size([1, 64, 55, 55])
ReLU            输出形状: torch.Size([1, 64, 55, 55])
MaxPool2d       输出形状: torch.Size([1, 64, 27, 27])
Conv2d          输出形状: torch.Size([1, 192, 27, 27])
ReLU            输出形状: torch.Size([1, 192, 27, 27])
MaxPool2d       输出形状: torch.Size([1, 192, 13, 13])
Conv2d          输出形状: torch.Size([1, 384, 13, 13])
ReLU            输出形状: torch.Size([1, 384, 13, 13])
Conv2d          输出形状: torch.Size([1, 256, 13, 13])
ReLU            输出形状: torch.Size([1, 256, 13, 13])
Conv2d          输出形状: torch.Size([1, 256, 13, 13])
ReLU            输出形状: torch.Size([1, 256, 13, 13])
MaxPool2d       输出形状: torch.Size([1, 256, 6, 6])
Flatten         输出形状: torch.Size([1, 9216])
Linear          输出形状: torch.Size([1, 4096])
ReLU            输出形状: torch.Size([1, 4096])
Dropout         输出形状: torch.Size([1, 4096])
Linear          输出形状: torch.Size([1, 4096])
ReLU            输出形状: torch.Size([1, 4096])
Dropout         输出形状: torch.Size([1, 4096])
Linear          输出形状: torch.Size([

---

### 4. 训练 AlexNet

> 由于训练 ImageNet 模型花费时间较长，因而，仅出于演示目的，我们使用 `Fashion-MNIST` 进行对 AlexNet 的训练。

为了适配 AlexNet，我们需要修改之前的 `load_fashion_mnist` 函数，加入 `Resize` 步骤。


In [7]:
import d2l_utils as d2l
from torch.utils import data

def load_data_alexnet(batch_size: int, resize: int = 224) -> tuple[data.dataloader, data.dataloader]:
    """加载 Fashion-MNIST 并调整尺寸以适配 AlexNet。"""
    return d2l.load_fashion_mnist(batch_size=batch_size, resize=resize)

# 建议配置
batch_size = 128
train_iter, test_iter = load_data_alexnet(batch_size)

Dataset check complete at: /home/august/deepseek/pytorch_study/temp/data


In [8]:
import d2l_utils as d2l
def train_alexnet(
    net: nn.Module,
    train_iter: data.DataLoader,
    test_iter: data.DataLoader,
    num_epochs: int,
    lr: float,
    device: torch.device
) -> None:
    """训练函数，用以演示 AlexNet 模型的训练过程。"""

    # 1. 初始化权重、搬运到 GPU
    def init_weights(m: nn.Module):
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            # 对输出层使用 Xavier 初始化（如果不接 ReLU）
            # 注意：这里的逻辑依赖于 net 的结构定义，如果最后一层是 net[-1]
            if m == net[-1]:
                nn.init.xavier_uniform_(m.weight)
            else:
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
    
    net.apply(init_weights)
    net.to(device)

    # 2. 优化器与损失函数
    # 按照论文配置 momentum 和 weight_decay
    optimizer = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=0.0005)
    loss_fn = nn.CrossEntropyLoss()

    print(f"在设备 {device} 上启动训练...")

    # 3. 训练循环
    for epoch in range(num_epochs):
        net.train()
        metric = d2l.Accumulator(3, device) # [train_loss_sum, train_acc_sum, n]

        for X, y in train_iter:
            optimizer.zero_grad()
            X, y = X.to(device), y.to(device)
            
            y_hat = net(X)
            l = loss_fn(y_hat, y)
            l.backward()
            optimizer.step()

            with torch.no_grad():
                metric.add(l * X.shape[0], d2l.count_correct_tensor(y_hat, y), X.shape[0])

        # 评估测试集
        train_loss = (metric[0]/metric[2]).item()
        train_acc = (metric[1]/metric[2]).item()
        test_acc = d2l.evaluate_accuracy_gpu(net, test_iter, device)
        print(f"Epoch {epoch+1} | Loss: {train_loss:.3f} | Train Acc: {train_acc:.3f} | Test Acc: {test_acc:.3f}")

# 运行示例
batch_size = 128 # 如果你的显卡显存小于 8GB，建议将 batch_size 设为 64 或 32
lr, num_epochs = 0.01, 10 # 建议先用较小的学习率观察收敛情况
device = d2l.get_default_device()

train_iter, test_iter = load_data_alexnet(batch_size)

net = get_alexnet_model()

train_alexnet(net, train_iter, test_iter, num_epochs, lr, device)

Dataset check complete at: /home/august/deepseek/pytorch_study/temp/data
在设备 cuda 上启动训练...
Epoch 1 | Loss: 0.609 | Train Acc: 0.780 | Test Acc: 0.866
Epoch 2 | Loss: 0.333 | Train Acc: 0.879 | Test Acc: 0.893
Epoch 3 | Loss: 0.280 | Train Acc: 0.898 | Test Acc: 0.904
Epoch 4 | Loss: 0.249 | Train Acc: 0.909 | Test Acc: 0.909
Epoch 5 | Loss: 0.228 | Train Acc: 0.917 | Test Acc: 0.905
Epoch 6 | Loss: 0.206 | Train Acc: 0.924 | Test Acc: 0.918
Epoch 7 | Loss: 0.193 | Train Acc: 0.929 | Test Acc: 0.919
Epoch 8 | Loss: 0.179 | Train Acc: 0.935 | Test Acc: 0.918
Epoch 9 | Loss: 0.167 | Train Acc: 0.938 | Test Acc: 0.924
Epoch 10 | Loss: 0.157 | Train Acc: 0.942 | Test Acc: 0.924


---

### 5. 细致梳理：AlexNet 的数学与逻辑

1.  **计算量与参数量**：
    *   AlexNet 虽然只有 8 层，但参数量高达 **6000 万**个。
    *   绝大部分参数集中在第一个全连接层：$256 \times 6 \times 6 \times 4096 \approx 3700$ 万。
2.  **ReLU 的胜利**：
    *   ReLU 的计算就是一行 `if x > 0`，而 Sigmoid 涉及复杂的指数运算。这使得 AlexNet 的训练速度比同规模的 Sigmoid 网络快 6 倍以上。
3.  **为什么第一层用 11x11 的大核？**
    *   在输入图片很大（224x224）时，需要一个足够大的窗口来捕捉最初的物体轮廓。

---

### 6. 关键工程暗知识 (Engineering Wisdom)

*   **显存预警**：AlexNet 的全连接层非常吃显存。如果你的显卡显存较小（如 4GB），请务必调小 `batch_size`（如 64 或 32）。
*   **学习率调优**：由于 AlexNet 比较深，建议初始学习率设小一点（如 0.01），并配合 Adam 优化器。

---

### 7. 总结 Checklist

*   **ReLU**：明白它为什么能取代 Sigmoid（解决梯度消失）。
*   **Dropout**：明白它在 AlexNet 这种大参数模型中的必要性。
*   **Resize**：掌握如何处理输入尺寸不匹配的问题。
*   **架构感**：记住 AlexNet “卷积-卷积-卷积卷积卷积-全连接”的节奏。

---


## 7.2 使用块的网络（VGG）

> 如果说 AlexNet 证明了“深”是有用的，那么 VGG 则确立了深度神经网络设计的**工业标准**：**模块化**。
> 
> VGG（由牛津大学视觉几何组 "Visual Geometry Group" 提出）不再纠结于每一层该用多大的核，而是发明了“VGG 块”的概念，通过重复堆叠相同的块来构建极深的网络。

### 1. 核心设计哲学：模块化

在 VGG 之前，网络设计像是在“做手工”，每一层的核大小、步幅都可能不同（如 AlexNet）。

**VGG 的创新**：
1.  **统一算子**：全线使用 $3 \times 3$ 卷积核和 $2 \times 2$ 最大汇聚层。
2.  **堆叠逻辑**：多个 $3 \times 3$ 卷积连续堆叠，其感受野等价于更大的卷积核（如 2 个 $3 \times 3$ 等价于 1 个 $5 \times 5$），但参数更少，非线性激活更多。
3.  **通道倍增**：每经过一个汇聚层，图像宽高减半，但输出通道数翻倍。

---

### 2. 构建 VGG块

In [9]:
import torch
from torch import nn, Tensor

def vgg_block(num_convs: int, in_channels: int, out_channels: int) -> nn.Sequential:
    """构建一个 VGG 块。

    每个 VGG 块包含若干个填充为 1 的 3x3 卷积层（ReLU 激活），
    最后接一个步幅为 2 的 2x2 最大汇聚层。

    Args:
        num_convs: 块内卷积层的数量。
        in_channels: 输入通道数。
        out_channels: 输出通道数。

    Returns:
        nn.Sequential: 组装好的 VGG 块。
        
        一个 VGG 块执行完后，输出形状变为：(Batch, C_{out}, H // 2, W // 2)
    """
    layers: list[nn.Module] = []
    for _ in range(num_convs):
        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
        layers.append(nn.ReLU())
        in_channels = out_channels # 后续卷积层的输入等于前一层的输出
    
    layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
    return nn.Sequential(*layers)

---

### 3. 构建 VGG-11 网络 

> 11 是指 8 卷积层 + 3 全连接层


In [10]:
def get_vgg_model(conv_arch: list[tuple[int, int]], in_channels: int = 1, num_classes: int = 10) -> nn.Sequential:
    """根据指定的架构列表构建 VGG 网络。

    Args:
        conv_arch: 
            - 架构列表，如 [(1, 64), (1, 128), (2, 256), (2, 512), (2, 512)]。
            - 元组内容：(卷积层数, 输出通道数)
        in_channels: 图像输入通道数。
        num_classes: 分类类别数。

    以 Fashion-MNIST 分类为例。

    Returns:
        nn.Sequential: 完整的 VGG 网络。
    """
    vgg_layers: list[nn.Module] = []
    
    # 1. 卷积块部分
    # 输入：(Batch, in_channels, 224, 224) 
    #   224 是因为 AlexNet 通常使用，感觉是一种习惯
    for (num_convs, out_channels) in conv_arch:
        vgg_layers.append(vgg_block(num_convs, in_channels, out_channels))
        in_channels = out_channels
        
    # 2. 全连接层部分
    # 设 经过 n 个 VGG 块
    # 则，输入为 (Batch, last_out_channels, 224 // (2 ** n), 224 // (2 ** n))
    side = 7 # 此处仅用以演示 224 // (2 ** 5)，写的是 VGG-11 的定义
    net = nn.Sequential(
        *vgg_layers,
        nn.Flatten(),   # -> (Batch, out_channels * side * side)
        nn.Linear(out_channels * side * side, 4096), nn.ReLU(), nn.Dropout(0.5), # -> (Batch, 4096)
        nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(0.5), # -> (Batch, 4096)
        nn.Linear(4096, num_classes) # (Batch, num_classes)
    )
    return net

# 定义标准的 VGG-11 架构
vgg11_arch = [(1, 64), (1, 128), (2, 256), (2, 512), (2, 512)]
net = get_vgg_model(vgg11_arch)

---

### 4. 训练 VGG-11


In [11]:
# 瘦身：因为参数量较大

# 将所有通道数除以 4，大幅减少计算量
ratio = 4
small_vgg_arch = [(pair[0], pair[1] // ratio) for pair in vgg11_arch]
net = get_vgg_model(small_vgg_arch)

# --- 验证维度 ---
X = torch.randn(size=(1, 1, 224, 224))
for blk in net:
    X = blk(X)
    print(f"{blk.__class__.__name__:15} 输出形状: {X.shape}")

Sequential      输出形状: torch.Size([1, 16, 112, 112])
Sequential      输出形状: torch.Size([1, 32, 56, 56])
Sequential      输出形状: torch.Size([1, 64, 28, 28])
Sequential      输出形状: torch.Size([1, 128, 14, 14])
Sequential      输出形状: torch.Size([1, 128, 7, 7])
Flatten         输出形状: torch.Size([1, 6272])
Linear          输出形状: torch.Size([1, 4096])
ReLU            输出形状: torch.Size([1, 4096])
Dropout         输出形状: torch.Size([1, 4096])
Linear          输出形状: torch.Size([1, 4096])
ReLU            输出形状: torch.Size([1, 4096])
Dropout         输出形状: torch.Size([1, 4096])
Linear          输出形状: torch.Size([1, 10])


In [12]:
# 该条不建议运行，感觉会比较慢
# 主要是学习新的 CNN 思想

import d2l_utils as d2l

# 超参数建议
lr, num_epochs, batch_size = 0.05, 10, 128
train_iter, test_iter = d2l.load_fashion_mnist(batch_size, resize=224)

# 执行训练，借用上面的函数
train_alexnet(net, train_iter, test_iter, num_epochs, lr, d2l.get_default_device())

Dataset check complete at: /home/august/deepseek/pytorch_study/temp/data
在设备 cuda 上启动训练...
Epoch 1 | Loss: 0.913 | Train Acc: 0.662 | Test Acc: 0.856
Epoch 2 | Loss: 0.350 | Train Acc: 0.871 | Test Acc: 0.883
Epoch 3 | Loss: 0.293 | Train Acc: 0.892 | Test Acc: 0.902
Epoch 4 | Loss: 0.259 | Train Acc: 0.904 | Test Acc: 0.910
Epoch 5 | Loss: 0.236 | Train Acc: 0.913 | Test Acc: 0.909
Epoch 6 | Loss: 0.216 | Train Acc: 0.920 | Test Acc: 0.914
Epoch 7 | Loss: 0.203 | Train Acc: 0.926 | Test Acc: 0.920
Epoch 8 | Loss: 0.192 | Train Acc: 0.929 | Test Acc: 0.920
Epoch 9 | Loss: 0.183 | Train Acc: 0.933 | Test Acc: 0.919
Epoch 10 | Loss: 0.172 | Train Acc: 0.936 | Test Acc: 0.923


### 5. 细致梳理：VGG 的数学与逻辑

1.  **为什么用 $3 \times 3$ 核？**
    *   两个 $3 \times 3$ 卷积层的参数量是 $2 \times (3 \times 3) = 18$。
    *   一个 $5 \times 5$ 卷积层的参数量是 $1 \times (5 \times 5) = 25$。
    *   **结论**：用多个小核代替大核，可以在保持感受野一致的情况下，**减少参数量**并**增加非线性层数**（每一层都有一个 ReLU）。
2.  **深度与复杂度的权衡**：
    *   VGG 证明了通过单纯地增加深度（VGG-16, VGG-19），模型的识别能力会稳步提升。
3.  **内存瓶颈**：
    *   VGG 的前两个全连接层占用了绝大部分参数量（约 1 亿个），这是现代架构（如 ResNet）致力于优化的方向。

---

### 6. 补充知识

`nn.Sequential` vs `nn.ModuleList` 的核心区别

| 特性 | `nn.Sequential` | `nn.ModuleList` |
| :--- | :--- | :--- |
| **执行逻辑** | **自动化**：内部自动实现了按顺序执行的 `forward`。 | **手动化**：只负责注册模块，`forward` 必须你自己写循环。 |
| **灵活性** | **低**：只能按严格的线性顺序执行。 | **高**：可以在循环中加入 `if/else`、跳跃连接（Skip Connection）等逻辑。 |
| **代码简洁度** | 极简，适合简单的堆叠。 | 略显繁琐，需要定义类。 |
| **用途场景** | 经典的 VGG 块、MLP 层。 | **更复杂的架构**（如 ResNet 的层管理、含有变长子模块的网络）。 |

为什么要有这两种组件？

1.  **参数注册 (Parameter Registration)**：
    两者最重要的共同点是都能**自动将列表内的参数注册到主模型中**。如果你只用 Python 原生的 `list`（例如 `self.layers = [nn.Linear(10, 10)]`），当你调用 `model.to(device)` 时，这些层**不会**被移动到 GPU。
    
2.  **VGG 的选择**：
    对于 VGG 这种“一条路走到黑”的线性架构，`nn.Sequential` 是最佳选择，因为它减少了样板代码。
    
3.  **ModuleList 的优势**：
    如果你在设计一个不确定深度的网络，或者在 `forward` 过程中需要用到中间层的输出（比如目标检测中的特征金字塔），`nn.ModuleList` 通过下标索引（`self.layers[i]`）的方式会非常方便。

---

### 7. 总结 Checklist

*   **VGG 块**：理解其内部构造（多个卷积 + 一个汇聚）。
*   **模块化设计**：明白如何通过改变 `arch` 列表来快速生成不同深度的网络。
*   **感受野等效性**：记住 $3 \times 3$ 堆叠的优势。
*   **显存意识**：知道为什么在实验中需要使用 `ratio` 进行缩放。

---


## 7.3 网络中的网络 (NiN)

> 如果说 VGG 解决了“如何模块化堆叠层”的问题，那么 NiN 则提出了一个更深刻的问题：卷积层之后一定要接笨重的全连接层吗？
>
> NiN 引入了两个革命性的概念：
> 
> - $1 \times 1$卷积层（用于增加非线性）和全局平均汇聚层（用于彻底取代全连接层）。

### 1. 设计动机

1.  **全连接层的弊端**：在 AlexNet 和 VGG 中，最后的全连接层占用了模型 90% 以上的参数量。这不仅容易导致过拟合，还极大地增加了显存负担。
2.  **局部结构的复杂性**：传统的卷积层只是做线性变换。NiN 认为，应该在每个像素位置应用一个“微型网络”来提取更复杂的特征。

---

### 2. 核心组件：NiN 块 (NiN Block)

一个 NiN 块由一个普通卷积层后接两个 $1 \times 1$ 卷积层组成。
*   **$1 \times 1$ 卷积的作用**：它充当了跨通道的全连接层，能够学习通道之间的复杂交互，同时完全保留空间结构。

---

### 3. 构建 NiN 块


In [15]:
import torch
from torch import nn, Tensor

def nin_block(
    in_channels: int, 
    out_channels: int, 
    kernel_size: int, 
    strides: int, 
    padding: int
) -> nn.Sequential:
    """构建一个 NiN 块。

    包含一个标准卷积层和两个 1x1 卷积层（充当全连接层）。
    
    Args:
        in_channels: 输入通道数。
        out_channels: 输出通道数。
        kernel_size: 第一层卷积的核大小。
        strides: 第一层卷积的步幅。
        padding: 第一层卷积的填充。

    Returns:
        nn.Sequential: 组装好的 NiN 块。
    """

    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, strides, padding),
        nn.ReLU(),
        # 第一个 1x1 卷积：增加非线性，不改变空间维度
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        # 第二个 1x1 卷积：进一步提取通道间特征
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU()
    )

---

### 4. NiN 网络架构

> NiN 彻底取消了最后那两个 4096 维的全连接层，改为使用**全局平均汇聚 (Global Average Pooling, GAP)**。


In [18]:
def get_nin_net(num_classes: int = 10) -> nn.Sequential:
    """构建 NiN 网络模型。"""
    net = nn.Sequential(
        # 块 1: 模拟 AlexNet 的第一层
        nin_block(1, 96, kernel_size=11, strides=4, padding=0),
        nn.MaxPool2d(3, stride=2),
        
        # 块 2: 增加深度
        nin_block(96, 256, kernel_size=5, strides=1, padding=2),
        nn.MaxPool2d(3, stride=2),
        
        # 块 3: 进一步提取特征
        nin_block(256, 384, kernel_size=3, strides=1, padding=1),
        nn.MaxPool2d(3, stride=2),
        nn.Dropout(0.5),
        
        # 块 4: 最后一块的输出通道数等于类别数
        nin_block(384, num_classes, kernel_size=3, strides=1, padding=1),
        
        # 全局平均汇聚层 (GAP)
        # 将 (Batch, 10, 5, 5) 变为 (Batch, 10, 1, 1)
        nn.AdaptiveAvgPool2d((1, 1)),
        
        # 展平后直接输出，不需要任何 Linear 层！
        nn.Flatten()
    )
    return net

In [19]:
# --- 验证维度流转 ---
X = torch.randn(size=(1, 1, 224, 224))
net = get_nin_net()
for layer in net:
    X = layer(X)
    print(f"{layer.__class__.__name__:20} 输出形状: {X.shape}")

Sequential           输出形状: torch.Size([1, 96, 54, 54])
MaxPool2d            输出形状: torch.Size([1, 96, 26, 26])
Sequential           输出形状: torch.Size([1, 256, 26, 26])
MaxPool2d            输出形状: torch.Size([1, 256, 12, 12])
Sequential           输出形状: torch.Size([1, 384, 12, 12])
MaxPool2d            输出形状: torch.Size([1, 384, 5, 5])
Dropout              输出形状: torch.Size([1, 384, 5, 5])
Sequential           输出形状: torch.Size([1, 10, 5, 5])
AdaptiveAvgPool2d    输出形状: torch.Size([1, 10, 1, 1])
Flatten              输出形状: torch.Size([1, 10])


---

### 5. 细致梳理：NiN 的数学与逻辑

1.  **$1 \times 1$ 卷积的本质**：
    *   它在空间维度上是“点乘”，在通道维度上是“全连接”。
    *   **优点**：引入了更多的非线性激活函数，增强了网络的表达能力，且参数量极低。
2.  **全局平均汇聚 (GAP) 的优势**：
    *   **参数压缩**：直接将最后一个特征图的每个通道求平均值作为分类得分。
    *   **正则化**：GAP 没有任何参数，天然防止了过拟合。
    *   **空间鲁棒性**：对输入图像的空间平移更加鲁棒。
3.  **架构对比**：
    *   **AlexNet/VGG**：卷积层提取特征 $\to$ 全连接层做分类。
    *   **NiN**：NiN 块提取高度抽象特征 $\to$ GAP 直接输出类别得分。

---

### 6. 关键工程暗知识 (Engineering Wisdom)

*   **`nn.AdaptiveAvgPool2d((1, 1))`**：这是实现 GAP 的标准 PyTorch 方式。无论输入特征图是 $5 \times 5$ 还是 $7 \times 7$，它都能自动计算步幅和核大小，将其压缩为 $1 \times 1$。
*   **训练建议**：由于 NiN 没有庞大的全连接层，它的训练速度通常比 AlexNet 快，且对显存的要求更低。建议学习率设为 0.1。

---

### 7. 补充知识

#### 7.1 核心定义

**消融实验 (Ablation Study)** 是指**从一个完整的、效果很好的系统中，移除某个特定的组件/特性（如某种层、某种损失函数、某个超参数），观察系统的表现会下降多少。**

如果移除它后效果大幅变差，说明这个组件是核心贡献者；如果效果没变，说明这个组件可能只是“凑数”的。

#### 7.2 消融实验 vs. 普通对比实验

*   **对比实验 (Comparative Study)**：通常是“横向”的。比如 A 算法和 B 算法哪个好？它是为了证明 **“我比别人强”**。
*   **消融实验 (Ablation Study)**：通常是“纵向”的。比如我的算法里有 3 个创新点，如果去掉创新点 1 会怎样？它是为了证明 **“我的每一个部分都有用”**。


#### 7.3 以 NiN 为例

> **实验设计**：将 NiN 最后的全局平均汇聚 (GAP) 换回 VGG 那样的两个 4096 维全连接层。

*   **系统 A (原版 NiN)**：使用 NiN 块 + GAP。
*   **系统 B (消融版)**：使用 NiN 块 + **全连接层**（相当于“消融”掉了 GAP 这个特性）。

**通过对比 A 和 B：**
1.  如果 B 的准确率大幅下降或产生严重过拟合，就证明了 **GAP（全局平均汇聚）在防止过拟合和特征抽象方面的卓越贡献**。
2.  这就是在验证 GAP 这个特定组件的价值。

#### 7.4 为什么要写消融实验？

在发表 AI 论文或进行复杂项目汇报时，消融实验是必不可少的，因为它能回答两个硬核问题：
1.  **拒绝“玄学”**：证明你的模型变好不是运气，而是因为这些设计的确科学。
2.  **定位核心**：弄清楚到底哪一步改进最关键，从而在资源有限时知道该保留什么。

**小结**：消融实验就是 **“拆零件，看机器还能不能跑、跑得顺不顺”** 的对比过程。

---

### 8. 总结 Checklist

*   **NiN 块**：理解 1x1 卷积如何充当“像素级全连接”。
*   **GAP**：理解为什么它可以取代全连接层。
*   **参数效率**：明白为什么 NiN 的参数量远小于 VGG。
*   **设计范式**：学会这种“纯卷积”架构的设计思路。

---
